In [12]:
import numpy as np

from typing import Union
from numpy.typing import ArrayLike

from sklearn.metrics import log_loss
from keras.losses import SparseCategoricalCrossentropy, CategoricalCrossentropy

In [13]:
# eps = epsilon
# It represents the lowest possible value in math
eps = np.finfo('float').eps
eps

np.float64(2.220446049250313e-16)

# Categorical Cross Entropy n=1

>L'entropie croisée (i.e. l'entropie entre deux distributions) permet de mesurer l'incertitude d'un modèle. 
>
>On recherche un modèle avec le moins d'incertitude possible. 
>
>C'est à dire un modèle qui renvoie les bonnes prédictions mais également avec une probablité associée très élevée.
>
>Autrement dit, quand notre modèle classifie correctement une image de chat, on veut également qu'il soit sur de lui.


> A contrario, au plus un modèle affirmera avec certitude des erreurs, au plus il sera pénalisé.
<br>

In [14]:
y_true = np.array([[False, True, False]]) # it means this sample belongs to the second class
y_pred = np.array([[0, .99, 0]]) # probability output of our model for each class

-np.sum(y_true*np.log(y_pred+eps))

np.float64(0.010050335853501225)

In [15]:
y_true = np.array([[False, True, False]])
y_pred = np.array([[.5, 0, .5]]) # worst case

# add eps to 0 because log(0) == np.inf
-np.sum(y_true*np.log(y_pred+eps))

np.float64(36.04365338911715)

# Categorical Cross Entropy n > 1

>Le calcul avec plusieurs observations consiste à simplement calculer la moyenne des entropies croisées inhérentes à chacune des prédictions

<br>

In [16]:
y_true = np.array([[0, 1, 0],
                   [0, 0, 1]])
                   
y_pred = np.array([[0.05, 0.95, .0],
                   [0.1, 0.1, 0.8]])

-np.sum(y_true*np.log(y_pred+eps), axis=1)


array([0.05129329, 0.22314355])

In [17]:
np.mean(-np.sum(y_true*np.log(y_pred+eps), axis=1))

np.float64(0.1372184228508799)

In [18]:
y_true = np.array([[0, 1, 0],
                   [0, 0, 1]])
                   
y_pred = np.array([[.05, .95, .0],
                   [.95, .025, .025]])  # big error

np.mean(-np.sum(y_true*np.log(y_pred+eps), axis=1))


np.float64(1.8700863742507388)

In [19]:
loss = CategoricalCrossentropy()
loss(y_true, y_pred).numpy()

np.float32(1.8700863)

# Sparse Categorical Cross Entropy

>La différence entre sparse et non sparse ne tient qu'à la manière dont sont présentés les y_true.
>
>La version sparse fournit un vecteur avec la classe prédite pour chacune des observations:

```py
y_true = ['chat', 'chien', 'chat', 'giraffe']
```


>La version non sparse fournit une représentation one hot encodée pour chacune des observations:

```py
y_true = [[1, 0, 0],
          [0, 1, 0],
          [1, 0, 0],
          [0, 0, 1]]
```

<br>

In [9]:
y_true = np.array([1, 2])

y_pred = np.array([[.05, .95, .0],
                   [.1, .1, .8]])

n_classes = np.max(y_true)+1

temp = np.zeros((y_true.shape[0], n_classes), dtype=int)
temp[np.arange(y_true.shape[0]), y_true] = 1
y_true = temp

np.mean(-np.sum(y_true*np.log(y_pred+eps), axis=1))


0.1372184228508799

<font size=10>$$cross\space entropy = -\frac{1}{N}\sum_{i=0}^{N-1} \sum_{k=0}^{K-1} y_{i, k} \log p_{i, k}$$</font>

where

n =  n sample

k = n class

In [10]:
def categorical_cross_entropy(y_true: np.array, y_pred: np.array, eps: float = np.finfo('float').eps) -> float:
    # TODO faster to clip or explicitly add eps. np.clip seems cleaner
    if y_pred.ndim == 1:
        return -np.mean(np.sum(y_true*np.log(y_pred+eps)))
    return -np.mean(np.sum(y_true*np.log(y_pred+eps), axis=1))


def sparse_categorical_cross_entropy(y_true: np.array, y_pred: np.array, **kwargs) -> float:
    n_classes = np.max(y_true)+1

    temp = np.zeros((y_true.shape[0], n_classes), dtype=int)
    temp[np.arange(y_true.shape[0]), y_true] = 1

    return categorical_cross_entropy(temp, y_pred, **kwargs)


In [11]:
y_true = np.array([1, 2])

y_pred = np.array([[.05, .95, .0],
                   [.1, .1, .8]])

sparse_categorical_cross_entropy(y_true, y_pred)

0.1372184228508799

In [12]:
y_true = np.array([1, 2])

y_pred = np.array([[.05, .95, .0],
                   [.1, .1, .8]])

sparse_categorical_cross_entropy(y_true, y_pred)

0.1372184228508799

In [13]:
# sklearn loss
print(log_loss(y_true, y_pred, labels=[0, 1, 2]))

# tf loss
loss = SparseCategoricalCrossentropy()
loss(y_true, y_pred).numpy()

0.13721842285088068


0.13721847534179688

# Cross Entropy, Entropy and KL divergence

>Il est également possible de retomber sur l'entropie croisée à l'aide de l'entropie
>
>et de la divergence de Kullback Leibler


<br>

>L'entropie ne prend en compte qu'une seule distribution, contrairement à l'entropie croisée.

<font size=10>$$entropy = -\sum_{i=1}^{m}p_i \log(p_i)$$</font>

> La divergence de Kullback Leibler quant à elle mesure l'entropie relative existante entre deux distributions.

<font size=10>$$D_{kl}(p||q) = \sum_{i=0} p_i\log\frac{p_i}{q_i}$$</font>

La somme de l'entropie et de cette divergence nous donne l'entropie croisée.

<font size=10>$$cross \space entropy=entropy(p)+D_{kl}(p||q)$$</font>

In [14]:
def get_p(y: np.array) -> np.array:
    return np.unique(y, return_counts=True)[1]/y.shape[0]


def entropy(p: ArrayLike, base: Union[str, int] = 'e') -> float:
    if base == 'e':
        log = np.log
    elif base == 2:
        log = np.log2

    return -np.sum(p*log(p))


def d_kl(p: ArrayLike, q: ArrayLike) -> float:
    # TODO is none get it 
    return np.sum(p*np.log(p/q))


In [15]:
y_true = np.array([1, 2, 3, 1, 2, 3])
y_pred = np.array([1, 2, 3, 2, 2, 1])

# Transform y_true and y_pred to probabilities for each class
p = get_p(y_true)
q = get_p(y_pred)


In [16]:
h = entropy(p)
div = d_kl(p, q)
crossh = h+div
print(f'crossh: {crossh:.5f}')
print(f'crossh: {categorical_cross_entropy(p, q):.5f}')

crossh: 1.19451
crossh: 1.19451
